In [1]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import requests

### Önce sayfamızı yükleyelim ve sayfada gezinelim

In [2]:
url = "https://en.wikipedia.org/wiki/World_population"

### Web sayfasının içeriğini metin formatında alın ve 'data' adlı bir değişkende sakla.

In [3]:
headers = {
   "User-Agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36"
}

data = requests.get(url, headers=headers)
soup = BeautifulSoup(data.text, "html.parser")


### Web sayfasındaki tüm HTML tablolarını bul

In [5]:
tables = soup.find_all("table")
len(tables)

27

### En yoğun nüfuslu 10 ülke tablosunu aradığımızı varsayalım, tablolar listesine bakıp her tablodaki verilere göre aradığımız doğru olanı bulabiliriz veya tabloda varsa tablo adını arayabiliriz ancak bu seçenek her zaman işe yaramayabilir.

In [12]:
for index,table in enumerate(tables):
    if ("10 most densely populated countries" in str(table)):
        table_index = index
print(table_index)

5


### Aşağıda, en yoğun nüfuslu 10 ülkeyi gösteren tablonun adını bulup bulamayacağınıza bir bakın.

In [13]:
print(tables[table_index].prettify())

<table class="wikitable sortable" id="mwBLU" style="text-align:right">
 <caption id="mwBLY">
  10 most densely populated countries
  <small id="mwBLc">
   (with population above 5
   <span id="mwBLg" typeof="mw:Entity">
   </span>
   million)
  </small>
  <sup about="#mwt477" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":":10"},"body":{"id":"mw-reference-text-cite_note-:10-106"}}' id="cite_ref-:10_106-0" rel="dc:references" typeof="mw:Extension/ref">
   <a href="#cite_note-:10-106" id="mwBLk">
    <span class="mw-reflink-text" id="mwBLo">
     <span class="cite-bracket" id="mwBLs">
      [
     </span>
     101
     <span class="cite-bracket" id="mwBLw">
      ]
     </span>
    </span>
   </a>
  </sup>
 </caption>
 <tbody id="mwBL0">
  <tr id="mwBL4">
   <th id="mwBL8" scope="col">
    Rank
   </th>
   <th id="mwBMA" scope="col">
    Country
   </th>
   <th id="mwBME" scope="col">
    Population
   </th>
   <th id="mwBMI" scope="col">
    Area
    <br id="mwBMM"/>
  

In [ ]:
population_data = pd.DataFrame(columns=["Rank", "Country", "Population", "Area", "Density"])

for row in tables[table_index].tbody.find_all("tr"):
    col = row.find_all("td")
    if col:
        rank = col[0].text.strip()
        country = col[1].text.strip()
        population = col[2].text.strip()
        area = col[3].text.strip()
        density = col[4].text.strip()
        # Yeni satır için geçici bir DataFrame oluştur
        new_row = pd.DataFrame([{"Rank": rank, "Country": country, "Population": population, "Area": area, "Density": density}])

        ## concat kullanalım
        population_data = pd.concat([population_data,new_row], ignore_index=True)

population_data

,Rank,Country,Population,Area,Density
0,1,Singapore,"5,921,231",719,"8,235"
1,2,Bangladesh,"165,650,475","148,460","1,116"
2,3,Palestine[note 3][102],"5,223,000","6,025",867
3,4,Taiwan[note 4],"23,580,712","35,980",655
4,5,South Korea,"51,844,834","99,720",520
5,6,Lebanon,"5,296,814","10,400",509
6,7,Rwanda,"13,173,730","26,338",500
7,8,Burundi,"12,696,478","27,830",456
8,9,Israel,"9,402,617","21,937",429
9,10,India,"1,389,637,446","3,287,263",423


## 2.yol BeautifulSoup ve read_html kullanarak HTML tablolarından veri kazıyıp bir DataFrame'e aktar


In [15]:
from io import StringIO

pd.read_html(StringIO(str(tables[5])), flavor="bs4")

[   Rank                 Country  Population  Area (km2)  Density (pop/km2)
 0     1               Singapore     5921231         719               8235
 1     2              Bangladesh   165650475      148460               1116
 2     3  Palestine[note 3][102]     5223000        6025                867
 3     4          Taiwan[note 4]    23580712       35980                655
 4     5             South Korea    51844834       99720                520
 5     6                 Lebanon     5296814       10400                509
 6     7                  Rwanda    13173730       26338                500
 7     8                 Burundi    12696478       27830                456
 8     9                  Israel     9402617       21937                429
 9    10                   India  1389637446     3287263                423]

In [16]:
population_data_read_html = pd.read_html(StringIO(str(tables[5])), flavor="bs4")[0]
population_data_read_html

,Rank,Country,Population,Area (km2),Density (pop/km2)
0,1,Singapore,5921231,719,8235
1,2,Bangladesh,165650475,148460,1116
2,3,Palestine[note 3][102],5223000,6025,867
3,4,Taiwan[note 4],23580712,35980,655
4,5,South Korea,51844834,99720,520
5,6,Lebanon,5296814,10400,509
6,7,Rwanda,13173730,26338,500
7,8,Burundi,12696478,27830,456
8,9,Israel,9402617,21937,429
9,10,India,1389637446,3287263,423


## read_html kullanarak HTML tablolarındaki verileri bir DataFrame'e aktarın
### HTML tablolarındaki verileri read_html kullanarak bir DataFrame'e aktarın

In [18]:
dataframe_list = pd.read_html(StringIO(str(data.text)), flavor="bs4")
len(dataframe_list)
dataframe_list[5]

,Rank,Country,Population,Area (km2),Density (pop/km2)
0,1,Singapore,5921231,719,8235
1,2,Bangladesh,165650475,148460,1116
2,3,Palestine[note 3][102],5223000,6025,867
3,4,Taiwan[note 4],23580712,35980,655
4,5,South Korea,51844834,99720,520
5,6,Lebanon,5296814,10400,509
6,7,Rwanda,13173730,26338,500
7,8,Burundi,12696478,27830,456
8,9,Israel,9402617,21937,429
9,10,India,1389637446,3287263,423


## İstediğimiz belirli tabloyu seçmek için `match` parametresini de kullanabiliriz. Eğer tablo, belirtilen metinle eşleşen bir dizge içeriyorsa, bu tablo okunacaktır.

In [22]:
heading = soup.find("h3", {"id": "Most_densely_populated_countries"})

table = heading.find_next("table")
df = pd.read_html(StringIO(str(table)))[0]
print(df.head())


   Rank                 Country  Population  Area (km2)  Density (pop/km2)
0     1               Singapore     5921231         719               8235
1     2              Bangladesh   165650475      148460               1116
2     3  Palestine[note 3][102]     5223000        6025                867
3     4          Taiwan[note 4]    23580712       35980                655
4     5             South Korea    51844834       99720                520
